In [ ]:
import json
from pathlib import Path
import os
import subprocess
import tempfile
import zipfile

from geojson_rewind import rewind
import geopandas as gpd
import numpy as np
import pandas as pd
import pycountry

In [ ]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_out_folder = base_dir + "data_out/templates/"

# shapes subfolder
data_in_shapes = data_in_folder + "shapes/"

# pcodes from ocha as requested by Abdul Karim
file_pcodes = "global_pcodes.csv"


In [ ]:
# countries for consultation of admin units from geo repo
country_list = [
    # "Kenya",
    "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]

iso3_list = [
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
    for country in country_list
]

iso2_list = [
    pycountry.countries.search_fuzzy(country)[0].alpha_2 if country != "Niger"
    else pycountry.countries.search_fuzzy(country)[1].alpha_2
    for country in country_list
]

# get pcodes from ocha for country list
pcodes_df = pd.read_csv(
    data_in_folder + file_pcodes,
    dtype=str,
    encoding="utf-8",
).query("Location in @iso3_list")


In [ ]:
# admin level selected
adm_level = "2"
# add selected admin with parent self-join
adm_level_pcodes_df = pcodes_df.query("`Admin Level` == @adm_level").merge(
    pcodes_df[["P-Code", "Name"]],
    how="left",
    left_on="Parent P-Code",
    right_on="P-Code",
    suffixes=(f"_{adm_level}", "_parent")
)

# add country name
adm_level_pcodes_df["country"] = country_list[0]


In [ ]:
# out columns
out_cols = [
    "country",
    "Location",
    f"P-Code_{adm_level}",
    f"Name_{adm_level}",
    "P-Code_parent",
    "Name_parent",
]

# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{adm_level}_pcode"
name_col = f"adm{adm_level}_name"

# output dataframe
out_df = adm_level_pcodes_df[out_cols].sort_values(
    [f"P-Code_{adm_level}"]
).rename(
    columns={
        "Location": "iso_code",
        f"P-Code_{adm_level}": pcode_col,
        f"Name_{adm_level}": name_col,
        "P-Code_parent": "parent_pcode",
        "Name_parent": "parent_name",
    }
)

# pcodes of analysis
pcodes_analysis = out_df[pcode_col].tolist()


In [ ]:
# save out template
out_df.to_excel(
    data_out_folder + f"{iso3_list[0]}_adm{adm_level}_Pcodes_Template.xlsx",
    index=False,
    sheet_name=f"{iso3_list[0]}_adm{adm_level}",
)


In [ ]:
# Get all files in shapes
files_in_shapes = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.is_file()
]

# identify shapes for country zip file
country_zip_shapes = [
    f for f in files_in_shapes if f"{iso3_list[0].lower()}_" in f
]

# Extract the country shape zip file into a temporary directory
with tempfile.TemporaryDirectory() as temp_dir:
    with zipfile.ZipFile(
        data_in_shapes + country_zip_shapes[0],
        'r',
    ) as zip_file:
        zip_file.extractall(temp_dir)

    # Displaying the contents of the temporary directory
    extracted_country_shapes = os.listdir(temp_dir)
    
    # identify desired admin level (assumes .shp file)
    country_admin_shape = [
        f for f in extracted_country_shapes
        if adm_level in f and f.endswith(".shp")
    ]

    # country_admin_shape geodataframe (assume CRS WGS84)
    gdf = gpd.read_file(
        os.path.join(temp_dir, country_admin_shape[0])
    ).to_crs('EPSG:4326').query(
        # avoid conflict shapes e.g. Lebanon adm3
        f"{pcode_col} in @pcodes_analysis"
    ).reset_index(drop=True)


In [ ]:
# Visvalingam’s weighted area in topojson
# use in mapshaper (% of removable points to retain)
shape_simp = 35
tol_map = f"{shape_simp}%"

# geojson output file name
geo_filename = f"geojson_{iso3_list[0]}_adm{adm_level}"

# run if shape_sim not 100 (i.e. no simplification)
if shape_simp != 100:

  # Create temporary input/output files for mapshaper simplification
  temp_input_file = tempfile.NamedTemporaryFile(suffix='.json')
  temp_out_file = tempfile.NamedTemporaryFile(
      suffix='.json', mode="w", encoding='utf-8'
  )

  # create temp geojson
  gdf.to_file(temp_input_file.name, driver="GeoJSON")

  # Use mapshaper to perform clean & topological simplifications
  subprocess.check_call([
    "mapshaper", "-i", temp_input_file.name,
    "-clean",
    "-simplify", tol_map, "keep-shapes",
    "-o", "format=geojson", temp_out_file.name
  ])

  # read back the simplified file from temp out
  gdf = gpd.read_file(temp_out_file.name).to_crs('EPSG:4326')

  # Cleanup temporary input files
  temp_input_file.close()
  temp_out_file.close()

  # modify geo_filename to indicate simplification
  geo_filename += f"_simpl{shape_simp}"


In [ ]:
# verify if names in shapes match template
dif_names = np.setdiff1d(
    out_df[name_col].tolist(),
    gdf[name_col].tolist(),
)

if len(dif_names) > 0:
    print("Names in template not matching gdf: use template")
    # drop name column in gdf and join names from template
    gdf = gdf.drop(columns=[name_col]).merge(
        out_df[[pcode_col, name_col]],
        how="left",
        on=pcode_col,
    )


In [ ]:
# create gejson output with rewinded shapes
geojson_out = rewind(
    json.loads(
        gdf[
            [pcode_col, name_col, "geometry"]
        ].to_json()
    ),
    rfc7946=False,
)

# export zip file with geojson
with zipfile.ZipFile(
    data_in_shapes + f"{geo_filename}.zip",
    'w',
    zipfile.ZIP_DEFLATED,
) as zip_file:
    # write geojson to zip file
    zip_file.writestr(
        f"{geo_filename}.geojson",
        json.dumps(geojson_out, ensure_ascii=False),
    )
